# Retail Sales Analytics Dashboard

| Field | Details |
|:---|:---|
| **Project** | Retail Sales Analytics & Customer Segmentation |
| **Dataset** | Online Retail II — UCI Machine Learning Repository |
| **Author** | Sanman |
| **Date** | March 2026 |

---

## Problem Statement

Retail businesses generate vast amounts of transactional data every day. Without structured analysis, this data remains an untapped resource. This project transforms raw e-commerce transaction records into actionable business intelligence.

The **Online Retail II** dataset contains sales transactions from a UK-based online retailer between 2009 and 2011, including invoice records, product descriptions, quantities, prices, customer IDs, and country information.

**Key challenges:**
- Significant missing customer data (~24% of transactions)
- Presence of cancellations, returns, and invalid entries
- No pre-existing customer segmentation or KPI metrics
- Need to uncover temporal, geographic, and behavioral sales patterns

The goal is to clean and transform the raw data, derive meaningful KPIs, and apply RFM (Recency, Frequency, Monetary) segmentation to classify customers into actionable groups that drive targeted marketing and retention strategies.

## Objectives

1. **Data Quality & Cleaning** — Handle missing values, remove cancellations, zero-price records, and negative quantities.
2. **Exploratory Data Analysis (EDA)** — Analyze revenue trends, top products, geographic sales distribution, and purchasing behavior by weekday and hour.
3. **Customer Analytics** — Identify high-value customers and analyze order value distribution.
4. **RFM Customer Segmentation** — Score and segment customers into Champions, Loyal Customers, Potential Loyalists, At Risk, and Lost Customers.
5. **Actionable Insights** — Derive business recommendations and export processed datasets.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

## 2. Data Loading

In [ ]:
df = pd.read_csv(r"C:\Users\Sanman\Downloads\Projects\retail-sales-analytics-dashboard\data\raw\online_retail_II.csv")
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:\n', df.columns.tolist())
print('\nData Types:\n', df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

> **Observation:** The dataset contains multiple data quality issues: missing customer IDs, negative quantities (returns), and zero or invalid prices. The majority of transactions originate from the United Kingdom. Data cleaning is required before analysis.

## 3. Missing Value Analysis

In [ ]:
df.isnull().sum()

> **Observation:** Missing values are concentrated in the `Customer ID` column (~24%), with minor gaps in `Description`. These will be handled appropriately during cleaning.

In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent.sort_values(ascending=False)

> **Observation:** Approximately 24% of transactions lack customer identifiers. These records will be excluded from customer-level segmentation but retained for overall sales trend analysis.

## 4. Data Cleaning

In [ ]:
clean_df = df.copy()

# Drop rows with missing critical fields
clean_df = clean_df.dropna(subset=['Description','Quantity','InvoiceDate','Price','Country'])

# Remove cancelled invoices (prefix 'C')
clean_df = clean_df[~clean_df['Invoice'].astype(str).str.startswith('C')]

# Remove invalid quantities and prices
clean_df = clean_df[clean_df['Quantity'] > 0]
clean_df = clean_df[clean_df['Price'] > 0]

# Subset with valid Customer IDs for customer-level analysis
customer_df = clean_df[clean_df['Customer ID'].notnull()]

print('Original dataset shape :', df.shape)
print('Cleaned dataset shape  :', clean_df.shape)
print('Customer dataset shape :', customer_df.shape)

## 5. Feature Engineering

In [ ]:
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])

clean_df['revenue']    = clean_df['Quantity'] * clean_df['Price']
clean_df['year']       = clean_df['InvoiceDate'].dt.year
clean_df['month']      = clean_df['InvoiceDate'].dt.month
clean_df['month_name'] = clean_df['InvoiceDate'].dt.month_name()
clean_df['weekday']    = clean_df['InvoiceDate'].dt.day_name()
clean_df['hour']       = clean_df['InvoiceDate'].dt.hour

print('-' * 42)
print(f"  Total Revenue  : GBP {clean_df['revenue'].sum():,.2f}")
print(f"  Total Orders   : {clean_df['Invoice'].nunique():,}")
print(f"  Total Customers: {clean_df['Customer ID'].nunique():,}")
print('-' * 42)

## 6. Exploratory Data Analysis

### 6.1 Monthly Sales Trend

In [ ]:
monthly_sales = clean_df.groupby(['year','month'])['revenue'].sum().reset_index()
monthly_sales = monthly_sales.sort_values(['year','month'])

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales['revenue'], marker='o', linewidth=2, color='steelblue')
plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')
plt.xlabel('Time Index (Month)')
plt.ylabel('Revenue (GBP)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **Observation:** Revenue shows clear seasonal fluctuations with a strong Q4 peak, consistent with the holiday shopping season.

### 6.2 Top 10 Products by Revenue

In [ ]:
top_products = (
    clean_df.groupby('Description')['revenue']
    .sum().sort_values(ascending=False).head(10)
)

plt.figure(figsize=(12, 6))
top_products.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

> **Observation:** A small number of products drive the majority of revenue (Pareto Principle). These top SKUs should be prioritized in inventory management and promotional planning.

### 6.3 Top 10 Countries by Revenue

In [ ]:
country_sales = (
    clean_df.groupby('Country')['revenue']
    .sum().sort_values(ascending=False).head(10)
)

plt.figure(figsize=(12, 6))
country_sales.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

> **Observation:** The United Kingdom dominates revenue. Germany, France, and EIRE follow at a significantly lower scale, presenting international expansion opportunities.

### 6.4 Sales by Weekday

In [ ]:
weekday_sales = clean_df.groupby('weekday')['revenue'].sum()
weekday_sales = weekday_sales.reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])

plt.figure(figsize=(10, 5))
weekday_sales.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Sales by Weekday', fontsize=14, fontweight='bold')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Observation:** Mid-week days (Tuesday to Thursday) show the highest customer activity. Sunday records notably lower sales, informing promotion scheduling and staffing.

### 6.5 Sales by Hour of Day

In [ ]:
hourly_sales = clean_df.groupby('hour')['revenue'].sum()

plt.figure(figsize=(10, 5))
hourly_sales.plot(kind='line', marker='o', color='darkorange', linewidth=2)
plt.title('Sales by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour (24h)')
plt.ylabel('Revenue (GBP)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **Observation:** Sales peak between 10:00 and 12:00 — optimal windows for flash promotions, push notifications, and email campaigns.

### 6.6 Top 10 Customers by Revenue

In [ ]:
top_customers = (
    clean_df.groupby('Customer ID')['revenue']
    .sum().sort_values(ascending=False).head(10)
)

plt.figure(figsize=(10, 6))
top_customers.plot(kind='bar', color='dodgerblue', edgecolor='black')
plt.title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Observation:** A small group of customers generates a disproportionately large share of revenue. Retaining these high-value customers through loyalty programs is critical.

### 6.7 Order Value Distribution

In [ ]:
order_value = clean_df.groupby('Invoice')['revenue'].sum()

plt.figure(figsize=(10, 5))
sns.histplot(order_value, bins=50, color='teal', kde=True)
plt.title('Order Value Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Order Value (GBP)')
plt.ylabel('Count')
plt.xlim(0, order_value.quantile(0.99))
plt.tight_layout()
plt.show()

> **Observation:** The distribution is heavily right-skewed. Most orders are low-to-moderate in value; a small number of high-value orders suggest the presence of bulk/wholesale buyers.

## 7. Export Cleaned Dataset

In [ ]:
output_dir = 'data/processed'
os.makedirs(output_dir, exist_ok=True)

clean_df.to_csv(os.path.join(output_dir, 'final_online_retail.csv'), index=False)
print(f'Cleaned dataset saved to: {output_dir}/final_online_retail.csv')

## 8. RFM Customer Segmentation

### 8.1 Compute RFM Metrics

In [ ]:
rfm_df = clean_df[clean_df['Customer ID'].notnull()]
snapshot_date = rfm_df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = rfm_df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice'    : 'nunique',
    'revenue'    : 'sum'
})
rfm.rename(columns={'InvoiceDate':'recency','Invoice':'frequency','revenue':'monetary'}, inplace=True)
rfm.head()

> **Observation:**
> - **Recency:** Days since last purchase (lower = better)
> - **Frequency:** Number of unique invoices (higher = more engaged)
> - **Monetary:** Total spend (higher = more valuable)

### 8.2 Score & Segment Customers

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

def segment_customer(row):
    if row['rfm_score'] in ['555','554','545','544','455']:
        return 'Champions'
    elif row['rfm_score'] in ['543','444','435','355','354']:
        return 'Loyal Customers'
    elif row['rfm_score'] in ['512','511','422','421']:
        return 'Potential Loyalists'
    elif row['rfm_score'] in ['311','312','221']:
        return 'At Risk'
    else:
        return 'Lost Customers'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
rfm.head()

### 8.3 Customer Segment Distribution

In [ ]:
segment_counts = rfm['segment'].value_counts()

plt.figure(figsize=(9, 5))
segment_counts.plot(kind='bar', color='slateblue', edgecolor='black')
plt.title('Customer Segmentation - Count per Segment', fontsize=14, fontweight='bold')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

> **Observation:** Champions and Loyal Customers form a solid core of engaged buyers. At Risk and Lost segments present clear re-engagement opportunities.

### 8.4 Revenue by Customer Segment

In [ ]:
segment_revenue = rfm.groupby('segment')['monetary'].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
segment_revenue.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Revenue by Customer Segment', fontsize=14, fontweight='bold')
plt.ylabel('Total Revenue (GBP)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

> **Observation:** Champions and Loyal Customers contribute the overwhelming majority of revenue, reinforcing the business case for robust retention and loyalty strategies.

## 9. Export Results

In [ ]:
rfm.to_csv(os.path.join(output_dir, 'rfm_customer_segmentation.csv'))
print(f'RFM segmentation saved to: {output_dir}/rfm_customer_segmentation.csv')

clean_df.columns = [
    'invoice','stockcode','description','quantity',
    'invoicedate','price','customer_id','country',
    'revenue','year','month','month_name','weekday','hour'
]
clean_df.to_csv(os.path.join(output_dir, 'final_online_retail.csv'), index=False)
print(f'Final cleaned dataset saved to: {output_dir}/final_online_retail.csv')

## 10. Overall Conclusion

---

This project delivered a comprehensive end-to-end retail sales analytics pipeline on the Online Retail II dataset.

### Data Quality
Approximately 24% of records lacked customer identifiers. After removing cancellations, invalid quantities, and zero-price entries, the dataset was reduced to a clean, analysis-ready form.

### Sales Trends
Monthly revenue exhibits strong Q4 seasonality, providing a reliable signal for advance inventory and marketing budget planning.

### Product & Geography
A small set of products drives the majority of revenue. The UK is the dominant market; Germany and France represent the highest-potential growth markets internationally.

### Temporal Behavior
Peak purchasing occurs mid-week (Tuesday to Thursday) between 10:00 and 12:00. These windows are optimal for scheduling promotions and resource allocation.

### RFM Customer Segmentation

| Segment | Recommendation |
|:---|:---|
| **Champions & Loyal Customers** | Reward with loyalty programs and early product access. |
| **Potential Loyalists** | Nurture with targeted offers and personalized communication. |
| **At Risk Customers** | Re-engage via win-back campaigns and churn surveys. |
| **Lost Customers** | Apply win-back strategies or reallocate budget to high-potential segments. |

### Strategic Takeaways
1. Concentrate retention investment on top-tier customers for maximum revenue protection.
2. Align campaigns with Q4 seasonal demand spikes.
3. Time promotions for mid-week, mid-morning windows.
4. Pursue strategic expansion into European markets.
5. Consolidate inventory around top-revenue SKUs.

---

*This analysis transforms raw transactional data into strategic business intelligence, enabling data-driven decisions across marketing, operations, customer retention, and international growth.*